# 09 — Diffusion for Tabular & Time Series

Diffusion models can be applied to structured data beyond images.

## Score-Based Generation Beyond Images

TabDDPM (Kotelnikov et al., 2022) extends DDPM to tabular data by:
1. Encoding mixed data types (continuous: Gaussian diffusion; categorical: multinomial diffusion)
2. Using a simple MLP rather than a UNet (no spatial structure in tabular data)
3. Applying Gaussian noise to continuous features; argmax-based discretisation for categoricals

For **time series** (TimeGrad, CSDI), the challenge is preserving temporal correlations during generation:
- CSDI (Tashiro et al., 2021) uses a score-based model conditioned on observed values to impute missing time steps — unifying imputation and forecasting
- The model conditions on the observed mask and produces probabilistic completions of the masked entries

Key insight: tabular diffusion produces better synthetic data than CTGAN on small datasets (per-column statistics are better preserved) but is slower to train and sample from.

In [ ]:
# TabDDPM: diffusion on a real tabular dataset
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

# Load tabular data
bc = load_breast_cancer()
X_raw = bc.data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_data = torch.tensor(X_scaled, dtype=torch.float32)
n_features = X_data.shape[1]
print(f'Dataset: {X_data.shape[0]} samples, {n_features} features')

# Noise schedule
T = 100
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# MLP noise predictor
class TabMLP(nn.Module):
    def __init__(self, n_features, T=100):
        super().__init__()
        self.time_embed = nn.Embedding(T, 64)
        self.net = nn.Sequential(
            nn.Linear(n_features + 64, 256), nn.SiLU(),
            nn.Linear(256, 256), nn.SiLU(),
            nn.Linear(256, 128), nn.SiLU(),
            nn.Linear(128, n_features)
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, self.time_embed(t)], dim=-1))

tab_net = TabMLP(n_features)
opt = torch.optim.Adam(tab_net.parameters(), lr=3e-4)

for step in range(3000):
    idx = torch.randint(0, len(X_data), (128,))
    x0 = X_data[idx]
    t = torch.randint(0, T, (128,))
    noise = torch.randn_like(x0)
    xt = alpha_bars[t].sqrt().unsqueeze(1)*x0 + (1-alpha_bars[t]).sqrt().unsqueeze(1)*noise
    loss = ((tab_net(xt, t) - noise)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

print('TabDDPM training complete')

In [ ]:
# Generate synthetic tabular data and evaluate quality
@torch.no_grad()
def tabddpm_sample(model, n_samples, T, alphas, alpha_bars, betas, n_features):
    x = torch.randn(n_samples, n_features)
    for t_val in range(T-1, -1, -1):
        t_b = torch.full((n_samples,), t_val, dtype=torch.long)
        eps = model(x, t_b)
        mean = (1/alphas[t_val].sqrt()) * (x - betas[t_val]/(1-alpha_bars[t_val]).sqrt() * eps)
        if t_val > 0:
            x = mean + betas[t_val].sqrt() * torch.randn_like(x)
        else:
            x = mean
    return x

synth = tabddpm_sample(tab_net, 200, T, alphas, alpha_bars, betas, n_features)
synth_np = synth.numpy()

# Statistical comparison
real_means = X_scaled[:200].mean(axis=0)
synth_means = synth_np.mean(axis=0)
real_stds = X_scaled[:200].std(axis=0)
synth_stds = synth_np.std(axis=0)

mean_err = np.abs(real_means - synth_means).mean()
std_err = np.abs(real_stds - synth_stds).mean()
print(f'Mean absolute error (column means): {mean_err:.4f}')
print(f'Mean absolute error (column stds): {std_err:.4f}')

# Compare first 4 feature distributions
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i in range(4):
    axes[0, i].hist(X_scaled[:200, i], bins=20, alpha=0.7, label='Real')
    axes[0, i].set_title(f'f{i} — Real')
    axes[1, i].hist(synth_np[:, i], bins=20, alpha=0.7, color='orange', label='Synth')
    axes[1, i].set_title(f'f{i} — TabDDPM')
plt.suptitle('Real vs TabDDPM — First 4 Features')
plt.tight_layout()
plt.savefig('/tmp/tabddpm.png', dpi=100, bbox_inches='tight')
plt.show()
print('TabDDPM comparison saved')